In [1]:
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Tuple, Optional
import numpy as np

# Define core type aliases
State = Any
Action = str
Policy = Callable[[State], Dict[Action, float]]

In [2]:
class Game:

    def start_state(self) -> State:
        raise NotImplementedError

    def successors(self, state: State) -> Dict[Action, State]:
        raise NotImplementedError

    def is_end(self, state: State) -> bool:
        raise NotImplementedError
    
    def player(self, state: State) -> str:
        raise NotImplementedError

    def utility(self, state: State) -> float:
        raise NotImplementedError
    

In [3]:
class Game1(Game):
    """Simple discrete game tree from Lecture 10."""
    def start_state(self) -> State:
        return "root"

    def successors(self, state: State) -> Dict[Action, State]:
        mapping = {
            "root": {"A": "A", "B": "B", "C": "C"},
            "A": {"1": "A1", "2": "A2"},
            "B": {"1": "B1", "2": "B2"},
            "C": {"1": "C1", "2": "C2"},
        }
        return mapping[state]

    def player(self, state: State) -> str:
        if state == "root":
            return "agent"
        if state in ["A", "B", "C"]:
            return "opp"
        raise ValueError(f"Invalid state:{state}")

    def is_end(self, state: State) -> bool:
        return state in ["A1", "A2", "B1", "B2", "C1", "C2"]

    def utility(self, state: State) -> float:
        utilities = {
            "A1": -50, "A2": 50,
            "B1": 1,   "B2": 3,
            "C1": -5,  "C2": 15,
        }
        return utilities[state]

@dataclass(frozen=True)
class HalvingState:
    n: int
    player: str

class HalvingGame(Game):
    """
    Halving Game: Two players take turns decrementing (-1) or halving (//2) a number.
    The player who reaches 0 wins.
    """
    def __init__(self, n: int):
        self.n = n

    def start_state(self) -> HalvingState:
        return HalvingState(n=self.n, player="agent")

    def successors(self, state: HalvingState) -> Dict[Action, HalvingState]:
        next_player = "opp" if state.player == "agent" else "agent"
        return {
            "decrement": HalvingState(n=state.n - 1, player=next_player),
            "half": HalvingState(n=state.n // 2, player=next_player),
        }

    def player(self, state: HalvingState) -> str:
        return state.player

    def is_end(self, state: HalvingState) -> bool:
        return state.n == 0

    def utility(self, state: HalvingState) -> float:
        assert state.n == 0
        # If it's agent's turn when n=0, it means opponent made the move that resulted in 0 -> Opponent wins!
        # If it's opp's turn, agent made the move that resulted in 0 -> Agent wins!
        return -1.0 if state.player == "agent" else 1.0

In [4]:
@dataclass(frozen=True)
class Step:
    action: Action
    state: State

@dataclass(frozen=True)
class Rollout:
    steps: List[Step]
    utility: float

def sample_action(policy_dist: Dict[Action, float]) -> Action:
    actions = list(policy_dist.keys())
    probs = list(policy_dist.values())
    return np.random.choice(actions, p=probs)

def simulate(game: Game, policies: Dict[str, Policy]) -> Rollout:
    steps = []
    state = game.start_state()
    while not game.is_end(state):
        player = game.player(state)
        action = sample_action(policies[player](state))
        state = game.successors(state).get(action)
        steps.append(Step(action, state))
    return Rollout(steps, game.utility(state))

def V_eval(game: Game, policies: Dict[str, Policy], state: State) -> float:
    if game.is_end(state):
        return game.utility(state)
    player = game.player(state)
    policy_dict = policies[player](state)
    utility = 0.0
    for action, prob in policy_dict.items():
        next_state = game.successors(state)[action]
        utility += prob * V_eval(game, policies, next_state)
    return utility

# Verification on Game1
def always_a_policy(state: State) -> Dict[Action, float]:
    return {"A": 1.0}

def random_policy(state: State) -> Dict[Action, float]:
    return {"1": 0.5, "2": 0.5}

policies = {"agent": always_a_policy, "opp": random_policy}
g1 = Game1()
print(f"Game1 Exact Evaluation Value:{V_eval(g1, policies, g1.start_state())}") # Expected: 0.0


Game1 Exact Evaluation Value:0.0


In [5]:
# Expectimax 
# Find agent's max utility given opp's policy

def V_exptmax(game: Game, opp_policy: Policy, state: State) -> float:
    if game.is_end(state):
        return game.utility(state)
    player = game.player(state)
    successors = game.successors(state)
    if player == 'agent':
        return max(V_exptmax(game, opp_policy, next_state) for next_state in successors.values())
    elif player == 'opp':
        utility = 0.0
        for action, prob in opp_policy(state).items():
            next_state = successors[action]
            utility += prob * V_exptmax(game, opp_policy, next_state)
        return utility
    raise ValueError(f"Unknown player:{player}")

print(f"Expectimax Value for Game1:{V_exptmax(g1, random_policy, g1.start_state())}") # Expected: 5.0 (Chooses C)


Expectimax Value for Game1:5.0


In [6]:
# Minimax
# Agent maximizes utility, opponent minimizes utility.

def V_minmax(game: Game, state: State) -> Tuple[float, Optional[Action]]:
    if game.is_end(state):
        return game.utility(state), None
    player = game.player(state)
    successors = game.successors(state)
    values = {action: V_minmax(game, next_state)[0] for action, next_state in successors.items()}
    if player == 'agent':
        best_action, value = max(values.items(), key = lambda item: item[1])
        return value, best_action
    elif player == 'opp':
        best_action, value = min(values.items(), key = lambda item: item[1])
        return value, best_action

    raise ValueError(f"Unknown player:{player}")

# Test on Game1 & HalvingGame
val, act = V_minmax(g1, g1.start_state())
print(f"Game1 Minimax Value:{val}, Best Action:{act}") # Expected: 1.0 (Chooses B)

hg = HalvingGame(n=11)
val_hg, act_hg = V_minmax(hg, hg.start_state())
print(f"HalvingGame(11) Minimax Result -> Value:{val_hg}, Optimal First Move:{act_hg}")

Game1 Minimax Value:1, Best Action:B
HalvingGame(11) Minimax Result -> Value:1.0, Optimal First Move:decrement


In [7]:
# Alpha-Beta Pruning implementation of Minimax

def V_alpha_beta(game: Game, state: State, alpha: float = -float('inf'), beta: float = float('inf')) -> Tuple[float, Optional[Action]]:
    if game.is_end(state):
        return game.utility(state), None
    player = game.player(state)
    successors = game.successors(state)
    best_action = None

    if player == 'agent':
        best_value = - float('inf')
        for action, next_state in successors.items():
            value, _ = V_alpha_beta(game, next_state, alpha, beta)
            if value > best_value:
                best_value = value
                best_action = action
            alpha = max(best_value, alpha)
            if alpha >= beta:
                break
        return best_value, best_action
    elif player == 'opp':
        best_value = float('inf')
        for action, next_state in successors.items():
            value, _ = V_alpha_beta(game, next_state, alpha, beta)
            if value < best_value:
                best_value = value
                best_action = action
            beta = min(best_value, beta)
            if alpha >= beta:
                break
        return best_value, best_action
    raise ValueError(f"Unknown player:{player}")

ab_val, ab_act = V_alpha_beta(g1, g1.start_state())
print(f"Alpha-Beta Value:{ab_val}, Best Action:{ab_act}")

Alpha-Beta Value:1, Best Action:B


In [8]:
# Limited depth search
Eval = Callable[[State], float]
def V_limited_depth_eval(game: Game, state: State, eval_fn: Eval, max_depth: int) -> float:
    if game.is_end(state):
        return game.utility(state)
    if max_depth == 0:
        return eval_fn(state)

    player = game.player(state)
    successors = game.successors(state)
    if player == 'agent':
        return max(V_limited_depth_eval(game, next_state, eval_fn, max_depth) for next_state in successors.values())
    elif player == 'opp':
        return min(V_limited_depth_eval(game, next_state, eval_fn, max_depth - 1) for next_state in successors.values())

    raise ValueError(f"Unknown player:{player}")

# Simple heuristic evaluation for HalvingGame
def simple_halving_eval(state: HalvingState) -> float:
    # Heuristic: smaller state values are generally better for the turn-taker
    return -float(state.n) if state.player == "agent" else float(state.n)

approx_val = V_limited_depth_eval(hg, hg.start_state(), simple_halving_eval, max_depth=3)
print(f"HalvingGame Depth-Limited (depth=3) Approx Value:{approx_val}")

HalvingGame Depth-Limited (depth=3) Approx Value:-1.0
